# Semana 7: Redes Recurrentes (RNN, LSTM, GRU) para NLP
## Notebook de Ejercicios (NB2) – Clasificación de Sentimiento con LSTM

**Propósito:** Aplicar LSTM a un problema real de análisis de sentimiento sobre reseñas de películas de IMDb, y comparar con el modelo de promedio de embeddings de la semana anterior.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Construir un modelo LSTM con embeddings entrenables en PyTorch.
- Entrenar el modelo y comparar su rendimiento con el modelo de promedio de embeddings.
- Visualizar las salidas ocultas para entender el comportamiento de la LSTM.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

# Scikit-learn para métricas
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Hugging Face datasets
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
except ImportError:
    DATASETS_AVAILABLE = False
    print("Nota: datasets no está instalado. Se instalará más adelante.")

# NLTK para tokenización
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga y Preprocesamiento del Dataset IMDb

Cargamos un subconjunto balanceado de IMDb reviews para el ejercicio.

In [ ]:
if not DATASETS_AVAILABLE:
    !pip install datasets
    from datasets import load_dataset

# Cargamos el dataset IMDb
print("Cargando dataset IMDb...")
imdb = load_dataset('imdb')

# Usamos un subconjunto para que el entrenamiento sea rápido
train_size = 2000  # 2000 reseñas para entrenamiento
test_size = 500    # 500 para prueba

# Tomar muestras
train_texts = imdb['train']['text'][:train_size]
train_labels = imdb['train']['label'][:train_size]

test_texts = imdb['test']['text'][:test_size]
test_labels = imdb['test']['label'][:test_size]

print(f"Entrenamiento: {len(train_texts)} reseñas")
print(f"Prueba: {len(test_texts)} reseñas")
print(f"\nProporción de positivos en entrenamiento: {np.mean(train_labels):.2%}")
print(f"Proporción de positivos en prueba: {np.mean(test_labels):.2%}")

# Mostrar un ejemplo
print("\n=== Ejemplo de reseña ===")
sentimiento = "POSITIVO" if train_labels[0] == 1 else "NEGATIVO"
print(f"Sentimiento: {sentimiento}")
print(f"Texto: {train_texts[0][:500]}...")

---
## 2. Construcción del Vocabulario y Codificación

### 2.1. Crear vocabulario a partir de los datos de entrenamiento

In [ ]:
def tokenize(text):
    """Tokenización simple por palabras."""
    return text.lower().split()

# Construir vocabulario
def build_vocab(texts, max_size=10000, min_freq=2):
    """Construye vocabulario con tokens especiales."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(tokenize(text))
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    
    # Añadir palabras que cumplen frecuencia mínima
    for word, count in word_counts.items():
        if count >= min_freq and len(vocab) < max_size:
            vocab[word] = len(vocab)
    
    return vocab

vocab = build_vocab(train_texts, max_size=10000, min_freq=2)
print(f"Tamaño del vocabulario: {len(vocab)}")
print(f"Primeras 10 palabras: {list(vocab.keys())[:10]}")

### 2.2. Convertir textos a índices con padding

In [ ]:
def encode_texts(texts, vocab, max_len=200):
    """Convierte textos a secuencias de índices con padding."""
    encoded = []
    for text in texts:
        tokens = tokenize(text)[:max_len]
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        # Padding
        indices += [vocab['<PAD>']] * (max_len - len(indices))
        encoded.append(indices)
    return torch.tensor(encoded)

max_len = 200  # longitud máxima de secuencia

X_train = encode_texts(train_texts, vocab, max_len)
y_train = torch.tensor(train_labels)

X_test = encode_texts(test_texts, vocab, max_len)
y_test = torch.tensor(test_labels)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")

### 2.3. Creación de DataLoaders

In [ ]:
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Número de lotes de entrenamiento: {len(train_loader)}")
print(f"Número de lotes de prueba: {len(test_loader)}")

---
## 3. Modelo de Promedio de Embeddings (Baseline)

Recordamos el modelo de la semana anterior para comparar.

In [ ]:
class AverageEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # Crear máscara para tokens que no son padding
        mask = (x != 0).unsqueeze(-1).float()  # (batch_size, seq_len, 1)
        
        # Suma de embeddings con máscara
        sum_embeddings = (embedded * mask).sum(dim=1)  # (batch_size, embed_dim)
        # Número de tokens reales por secuencia
        lengths = mask.sum(dim=1)  # (batch_size, 1)
        # Evitar división por cero
        pooled = sum_embeddings / (lengths + 1e-8)  # (batch_size, embed_dim)
        
        h = self.dropout(self.relu(self.fc1(pooled)))
        out = self.fc2(h)
        return out

model_avg = AverageEmbeddingModel(len(vocab), embed_dim=100, hidden_dim=64)
model_avg.to(device)
print("Modelo de promedio de embeddings creado.")

---
## 4. Modelo LSTM con Embeddings Entrenables

Construimos un modelo LSTM para clasificación de sentimiento.

In [ ]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_layers=1, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Usamos el último estado oculto (de la última capa)
        last_hidden = hidden[-1]  # (batch_size, hidden_dim)
        
        out = self.fc(self.dropout(last_hidden))
        return out

model_lstm = LSTMSentiment(len(vocab), embed_dim=100, hidden_dim=64, num_layers=1)
model_lstm.to(device)

# Contar parámetros
total_params_lstm = sum(p.numel() for p in model_lstm.parameters())
print(f"Modelo LSTM creado. Total de parámetros: {total_params_lstm:,}")

---
## 5. Entrenamiento de Ambos Modelos

### 5.1. Función de entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    for epoch in range(epochs):
        # Entrenamiento
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_loss = epoch_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Evaluación en test
        model.eval()
        test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        test_loss /= len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return train_losses, train_accs, test_losses, test_accs

In [ ]:
# Entrenar modelo de promedio
print("Entrenando modelo de promedio de embeddings...")
train_losses_avg, train_accs_avg, test_losses_avg, test_accs_avg = train_model(
    model_avg, train_loader, test_loader, epochs=10, lr=0.001
)

In [ ]:
# Entrenar modelo LSTM
print("\nEntrenando modelo LSTM...")
train_losses_lstm, train_accs_lstm, test_losses_lstm, test_accs_lstm = train_model(
    model_lstm, train_loader, test_loader, epochs=10, lr=0.001
)

---
## 6. Comparación de Resultados

### 6.1. Evolución del accuracy durante entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(test_accs_avg, 'b-o', label='Promedio embeddings')
axes[0].plot(test_accs_lstm, 'r-o', label='LSTM')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy en test')
axes[0].set_title('Comparación de Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(test_losses_avg, 'b-o', label='Promedio embeddings')
axes[1].plot(test_losses_lstm, 'r-o', label='LSTM')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Pérdida en test')
axes[1].set_title('Comparación de Pérdida')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print("=== RESULTADOS FINALES ===")
print(f"Modelo de promedio - Accuracy final en test: {test_accs_avg[-1]:.4f}")
print(f"Modelo LSTM - Accuracy final en test: {test_accs_lstm[-1]:.4f}")

mejora = (test_accs_lstm[-1] - test_accs_avg[-1]) * 100
print(f"Mejora de LSTM sobre promedio: {mejora:+.2f}%")

---
## 7. Visualización de las Salidas Ocultas de la LSTM

Tomamos algunas oraciones de prueba y visualizamos los estados ocultos a lo largo del tiempo.

In [ ]:
# Función para extraer estados ocultos
def extract_hidden_states(model, X_batch):
    model.eval()
    with torch.no_grad():
        # Embedding
        embedded = model.embedding(X_batch)
        # Pasar por LSTM y obtener todas las salidas
        lstm_out, (hidden, cell) = model.lstm(embedded)
    return lstm_out.cpu().numpy()

# Tomar un lote de prueba
X_sample, y_sample = next(iter(test_loader))
X_sample = X_sample[:4].to(device)  # 4 ejemplos

# Extraer estados ocultos
hidden_states = extract_hidden_states(model_lstm, X_sample)
print(f"Forma de los estados ocultos: {hidden_states.shape} (batch_size, seq_len, hidden_dim)")

# Visualizar la evolución de las primeras dimensiones para un ejemplo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx in range(4):
    hidden_sequence = hidden_states[idx]  # (seq_len, hidden_dim)
    # Mostrar las primeras 10 dimensiones
    for dim in range(5):
        axes[idx].plot(hidden_sequence[:, dim], label=f'dim {dim}')
    axes[idx].set_title(f'Ejemplo {idx+1} - Sentimiento: {"POS" if y_sample[idx]==1 else "NEG"}')
    axes[idx].set_xlabel('Paso temporal')
    axes[idx].set_ylabel('Valor del estado oculto')
    axes[idx].legend()
    axes[idx].grid(True)

plt.tight_layout()
plt.show()

### 7.1. Visualización con PCA de los estados ocultos finales

In [ ]:
from sklearn.decomposition import PCA

# Obtener estados ocultos finales para todos los ejemplos de test
model_lstm.eval()
final_hidden_states = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        embedded = model_lstm.embedding(X_batch)
        lstm_out, (hidden, cell) = model_lstm.lstm(embedded)
        last_hidden = hidden[-1].cpu().numpy()
        final_hidden_states.append(last_hidden)
        all_labels.append(y_batch.numpy())

final_hidden_states = np.vstack(final_hidden_states)
all_labels = np.concatenate(all_labels)

# Reducir a 2D con PCA
pca = PCA(n_components=2)
hidden_pca = pca.fit_transform(final_hidden_states)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(hidden_pca[:, 0], hidden_pca[:, 1], c=all_labels, cmap='bwr', alpha=0.6)
plt.colorbar(scatter, label='Sentimiento (0=Negativo, 1=Positivo)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Estados ocultos finales de LSTM proyectados con PCA')
plt.show()

---
## 8. Experimentación Adicional

### 8.1. Prueba con diferentes configuraciones de LSTM

Podemos experimentar con:
- Número de capas LSTM
- Dimensionalidad del estado oculto
- LSTM bidireccional

In [ ]:
class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_layers=1, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, 
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 para bidireccional
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Concatenar los últimos estados de ambas direcciones
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]
        hidden_concat = torch.cat((hidden_forward, hidden_backward), dim=1)
        out = self.fc(self.dropout(hidden_concat))
        return out

# Descomentar para probar
# model_bilstm = BiLSTMSentiment(len(vocab))
# model_bilstm.to(device)
# ... entrenar ...

---
## 9. Conclusiones

En este notebook hemos aplicado LSTM a un problema real de análisis de sentimiento:

✔️ **Modelo LSTM**: Implementamos y entrenamos un modelo LSTM con embeddings entrenables.

✔️ **Comparación con promedio de embeddings**:
- LSTM captura el orden de las palabras, lo que mejora el rendimiento.
- La mejora puede ser de 1-3% en accuracy respecto al modelo de promedio.

✔️ **Visualización de estados ocultos**:
- Los estados ocultos evolucionan a lo largo de la secuencia.
- La proyección PCA muestra cierta separación entre clases.

✔️ **Experimentación**: Posibilidad de probar LSTM bidireccional y otras configuraciones.

**Lección clave**: Las LSTM son superiores a los modelos de promedio cuando el orden de las palabras importa. Sin embargo, son más lentas de entrenar y tienen más parámetros.

---
**Fin del Notebook de Ejercicios - Semana 7 (NLP)**